# Basic run

In [1]:
%load_ext autoreload
%autoreload all

In [2]:
import polars as pl
import src.configs as configs
import src.graph_fct as graph_fct
import src.utils as ut
import pickle


# 0. load basic

In [3]:
df_relation = pl.read_parquet(configs.GraphConfig.relation_path)
g_is_a = graph_fct.get_is_a_graph(df_relation)
df_concept_hug = pl.read_parquet(configs.GraphConfig.concept_path)


# 1. compute the max depth

In [4]:
df_concept_hug_w_depth = graph_fct.get_max_min_depth_snomed(G=g_is_a,df_all_concepts=df_concept_hug)
df_concept_hug_w_depth.write_parquet(configs.ProcessedGraph.concept_w_depth)

Calculating depths: 100%|██████████| 447421/447421 [00:15<00:00, 29598.64it/s]
c:\Users\yy\Desktop\codes_PhD\graph_tokenizer\src\graph_fct.py:66: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  .with_columns(
c:\Users\yy\Desktop\codes_PhD\graph_tokenizer\src\graph_fct.py:69: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  .with_columns(


# 2. propagate concepts

In [5]:
concept_porpagate = ut.ConceptPropagate(df_relation, df_concept_hug_w_depth)
mapped_candidate_rel_dist_prop = concept_porpagate.build_all_distance_and_relations()
mapped_candidate_rel_dist_prop.write_parquet(configs.ProcessedGraph().mapped_candidate_rel_dist_prop)

100%|██████████| 46150/46150 [00:00<00:00, 46857.71it/s]


In [24]:
mapped_candidate_rel_dist_prop = pl.read_parquet(configs.ProcessedGraph().mapped_candidate_rel_dist_prop)


In [25]:
mapped_candidate_rel_dist_prop

src.id,dst.id,relation,distance
str,str,str,i32
"""204508009+306963008""","""308490002""","""PATHOLOGICAL_PROCESS""",1
"""117033003:116686009=309201001,…","""15220000""","""IS_A""",4
"""28944009""","""138875005""","""CAUSATIVE_AGENT""",8
"""69527006:116686009=119361006""","""138875005""","""HAS_SPECIMEN""",5
"""271903000:{246090004=860603002…","""127325009""","""ASSOCIATED_FINDING""",4
…,…,…,…
"""1156530009:{363703001=36367600…","""249597005""","""HAS_FOCUS""",6
"""117034009:116686009=309201001,…","""129264002""","""METHOD""",2
"""50485007""","""246915008""","""IS_A""",3


# 2. get is_a parent:child reachable map

In [6]:
candidate_reachable_child_map = ut.precompute_candidate_reachable_child_map(
                                    g_is_a,
                                    mapped_candidate_rel_dist_prop["dst.id"].unique(),
                                    )

with open(configs.ProcessedGraph().candidate_is_a_reachable_dict, "wb") as f:
    pickle.dump(candidate_reachable_child_map, f)

Precomputing reachable child candidates: 100%|██████████| 75669/75669 [00:18<00:00, 4100.98it/s]
